# 01 — Explore AOI/configs + synthetic skeleton → smoke point table

**SYNTHETIC / SMOKE ONLY — NOT real labels.** Every feature and label below is
generated inline with seed 42 for a <5-minute CPU smoke test. Real labels live at
`data/labels/hyd_points.geojson` (QGIS, human-only) and are NEVER touched here.

Order: (a) clone/install repo package → (b) load configs (AOI Hyderabad, Sentinel spec)
→ (c) real skeleton + member-overlap logic (`geoeco.labels.pipeline`) → synthetic
point table. Hands off to `02_train_classical.ipynb` for the training smoke.

Wall time: ~1–2 min on a free Kaggle CPU notebook. Still needs real labels / GPU: everything
under “REAL-LABELS next steps” in notebook 02 (M1–M5 full RF + M6 UNet).

In [ ]:
# Cell 1 — Kaggle/CPU setup: clone repo if needed, make `geoeco` importable (idempotent).
import subprocess
import sys
from pathlib import Path


def find_repo() -> Path:
    cands = [Path.cwd(), Path.cwd() / "PrithviBodh",
             Path("/kaggle/working/PrithviBodh"), Path("/kaggle/working")]
    for c in cands:
        if (c / "configs" / "aoi" / "hyderabad.yaml").exists():
            return c
    subprocess.run(["git", "clone", "https://github.com/veeresh0804/PrithviBodh.git"],
                   check=True, cwd=str(Path.cwd()))
    return Path.cwd() / "PrithviBodh"


REPO = find_repo()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print("REPO =", REPO)

# Minimal smoke deps (preinstalled on Kaggle; pip is a no-op fallback).
import importlib
for _mod, _pip in [("sklearn", "scikit-learn"), ("mlflow", "mlflow"),
                   ("yaml", "pyyaml"), ("pandas", "pandas"),
                   ("numpy", "numpy"), ("pyarrow", "pyarrow")]:
    try:
        importlib.import_module(_mod)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", _pip], check=True)

import mlflow
import numpy
import pandas
import sklearn
print("sklearn", sklearn.__version__, "| mlflow", mlflow.__version__,
      "| pandas", pandas.__version__, "| numpy", numpy.__version__)

In [ ]:
# Cell 2 — Load configs: AOI Hyderabad + Sentinel spec (+ seed gate + RF reference).
from geoeco.utils.config import load_yaml_config, require_keys

aoi = load_yaml_config(REPO / "configs" / "aoi" / "hyderabad.yaml")
require_keys(aoi, ["bounds_wgs84", "center", "crs", "resolution_m"], name="hyderabad.yaml")
sent = load_yaml_config(REPO / "configs" / "data" / "sentinel.yaml")
require_keys(sent, ["years", "seasons", "sentinel2", "sentinel1", "stack_per_season"],
             name="sentinel.yaml")
lab = load_yaml_config(REPO / "configs" / "labels" / "labelling.yaml")
cv = load_yaml_config(REPO / "configs" / "eval" / "spatial_cv.yaml")
assert lab["seed"] == cv.get("seed") == 42, f"seed gate failed: {lab['seed']} vs {cv.get('seed')}"
rf = load_yaml_config(REPO / "configs" / "model" / "rf.yaml")

print("AOI bounds (WGS84):", aoi["bounds_wgs84"])
print("AOI CRS:", aoi["crs"], "| resolution_m:", aoi["resolution_m"], "| center:", aoi["center"])
print("Sentinel years:", sent["years"], "| seasons:", list(sent["seasons"]))
print("S2 bands:", sent["sentinel2"]["bands"], "| indices:", sent["sentinel2"]["indices"])
print("S1 feats:", sent["sentinel1"]["features"], "| stack_per_season:", sent["stack_per_season"])
print("seed:", lab["seed"], "| splits:", lab["splits"], "| folds:", lab["folds"])
print("RF reference (real run): n_estimators =", rf["rf"]["n_estimators"],
      "| smoke override in nb 02: n_estimators = 10")

In [ ]:
# Cell 3 — SYNTHETIC skeleton geometry via the REAL labelling pipeline (labels stay EMPTY).
# build_skeleton() + select_overlap() are the member double-labelling logic from
# geoeco/labels/pipeline.py. No DW/WorldCover is read; no label is filled (rule 1).
from geoeco.labels.pipeline import build_skeleton, select_overlap

pts, counts = build_skeleton()
n_over = len(select_overlap(pts, float(lab["overlap_fraction"]), int(lab["seed"])))
print("skeleton counts:", counts)
print("overlap (double-labelled) points:", n_over)

# Guards: labels MUST be empty; test/train blocks MUST be disjoint (no leakage).
assert all(p["label"] is None for p in pts), "skeleton labels must stay empty"
test_blocks = {p["block"] for p in pts if p["split"] == "test"}
train_blocks = {p["block"] for p in pts if p["split"] == "train"}
assert not (test_blocks & train_blocks), f"leakage: {sorted(test_blocks & train_blocks)}"
print("test_blocks:", sorted(test_blocks), "| train_blocks:", sorted(train_blocks))
print("sample:", [{k: p[k] for k in ("id", "block", "split", "fine_block", "overlap_id")}
      for p in pts[:3]])

In [ ]:
# Cell 4 — SYNTHETIC/SMOKE point table (seed 42). Random features + random labels.
# Columns follow geoeco.models.classical.feature_columns('M3') = 60ch contract
# (14 optical + 4 SAR + 2 terrain) x 3 seasons. `synthetic=True` marks every row so
# this table can NEVER be confused with real labels.
import numpy as np
import pandas as pd

from geoeco.models.classical import feature_columns

SMOKE_SEED = 42
PER_BLOCK = 60  # 60 pts x 5 blocks (B0-B4) = 300 rows -> GroupKFold(5) needs >= 5 groups
cols = feature_columns("M3")
assert len(cols) == 60, f"M3 contract drift: {len(cols)} cols"

rng = np.random.default_rng(SMOKE_SEED)
sub = []
for b in sorted({p["block"] for p in pts}):
    sub += [p for p in pts if p["block"] == b][:PER_BLOCK]
n = len(sub)
X = rng.normal(size=(n, len(cols))).astype(np.float32)
y = rng.integers(0, 6, size=n)  # SYNTHETIC labels 0..5 (NOT human, NOT DW/WorldCover)
X += y[:, None].astype(np.float32) * 0.3  # weak class signal so smoke F1 is non-degenerate

df = pd.DataFrame(X, columns=cols)
df["label"] = y
df["block_id"] = [p["block"] for p in sub]
df["synthetic"] = True
print("shape:", df.shape, "| classes:", sorted(df['label'].unique().tolist()))
print("per-block:", df["block_id"].value_counts().to_dict())
print(df.head(2).to_string())

SMOKE_PARQUET = REPO / "smoke_points_M3.parquet" if "REPO" in dir() else Path("smoke_points_M3.parquet")
try:
    df.to_parquet(SMOKE_PARQUET, index=False)
    print("saved:", SMOKE_PARQUET)
except Exception as e:  # pyarrow missing etc. — nb 02 regenerates inline anyway
    print("parquet save skipped:", type(e).__name__)

In [ ]:
# Cell 5 — Spatial-group sanity: 5-fold GroupKFold over skeleton blocks (no leakage).
from sklearn.model_selection import GroupKFold

from geoeco.features.sampling import assert_no_leakage

groups = df["block_id"].to_numpy()
assert len(set(groups.tolist())) >= 5, "need >= 5 blocks for GroupKFold(5)"
for fold, (tr, te) in enumerate(GroupKFold(n_splits=5).split(df, df["label"], groups)):
    assert_no_leakage(groups[tr].tolist(), groups[te].tolist())
    print(f"fold {fold}: train={len(tr)} test={len(te)} test_block={sorted(set(groups[te].tolist()))}")
print("01 DONE — smoke point table ready; open 02_train_classical.ipynb for the training smoke.")